# Facebook Political Advertising Analytics at Scale

This notebook analyzes large-scale Facebook political advertising data in Australia (2020–2024) using PySpark.

The dataset contains over **39 million ad snapshots**, cleaned into **482k unique ads** for analysis.

Key analyses include:

- Data cleaning and preprocessing
- Keyword frequency analysis
- Keyword spend analysis
- Top political funders
- Time-based advertising trends
- COVID campaign analysis
- CPM cost efficiency analysis

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, array, explode, split, lower, regexp_replace,
    count, sum
)

import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder.appName("FacebookAdsAnalysis").getOrCreate()

## Load Facebook Ad Dataset

The raw dataset contains millions of JSON records stored in HDFS.  
Each record represents a Facebook political advertisement snapshot.

In [ ]:
datafile = spark.read.option(
    "multiLine",
    "false"
).json("hdfs:///data/ProjectDatasetFacebookAU/fbadsAU/*")

datafile.cache()

print("Total Raw Records:", datafile.count())

## Data Cleaning

Remove advertisements with missing:

- funding entity
- page name
- ad text

In [ ]:
ads = datafile.filter(
    (col("funding_entity").isNotNull()) &
    (col("page_name").isNotNull()) &
    ((col("ad_creative_bodies").isNotNull()) | (col("ad_creative_body").isNotNull()))
)

ads = ads.dropDuplicates(["id"])

## Normalize Ad Text Fields

Some ads store text in arrays while others use single text fields.
We normalize them into a single column.

In [ ]:
ads = ads.withColumn(
    "text_array",
    when(col("ad_creative_bodies").isNotNull(), col("ad_creative_bodies"))
    .otherwise(array(col("ad_creative_body")))
)

In [ ]:
ads_flat = ads.select(
    "ad_creation_time",
    explode("text_array").alias("ad_text"),
    "page_name",
    "funding_entity",
    col("spend.upper_bound").cast("int").alias("spend_upper"),
    col("impressions.upper_bound").cast("int").alias("impressions_upper")
)

ads_clean = ads_flat.filter(
    (col("ad_text") != "") & col("ad_text").isNotNull()
)

## Keyword Tokenization

Ad text is tokenized into individual words to analyze messaging patterns.

In [ ]:
ads_split = ads_clean.withColumn(
    "word",
    explode(split(lower(col("ad_text")), "\\W+"))
)

In [ ]:
word_filter = ads_split.filter(
    (col("word") != "") &
    (~col("word").isin("the", "and", "to", "a", "of", "is", "in", "on", "for"))
)

In [ ]:
word_count = word_filter.groupBy("word") \
    .count() \
    .orderBy("count", ascending=False)

word_count.show(20)

## Keyword Spend Analysis

We join tokenized words with advertisement spending to identify
which political keywords receive the most funding.

In [ ]:
ads_split_clean = ads_split.drop("spend_upper")

spend_data = ads_clean.select(
    "ad_text",
    col("spend_upper").alias("spend_amt")
)

join_spend = ads_split_clean.join(
    spend_data,
    on="ad_text",
    how="left"
)

In [ ]:
join_spend_clean = join_spend.filter(
    (col("word") != "") &
    (col("spend_amt").isNotNull())
)

word_spend = join_spend_clean.groupBy("word") \
    .sum("spend_amt") \
    .withColumnRenamed("sum(spend_amt)", "total_spend") \
    .orderBy("total_spend", ascending=False)

word_spend.show(20)

## Top Political Funders

In [ ]:
investors = ads_clean.groupBy("funding_entity") \
    .sum("spend_upper") \
    .withColumnRenamed("sum(spend_upper)", "total_spend") \
    .orderBy("total_spend", ascending=False)

investors.show(10)

In [ ]:
## CPM Cost Efficiency Analysis

In [ ]:
reach_df = ads_clean.select(
    "ad_text",
    "funding_entity",
    "page_name",
    col("spend_upper").cast("double").alias("spend"),
    col("impressions_upper").cast("double").alias("impressions")
).filter(
    col("spend").isNotNull() &
    col("impressions").isNotNull() &
    (col("impressions") > 0)
)

reach_df = reach_df.withColumn(
    "cpm",
    (col("spend") / col("impressions")) * 1000
)

In [ ]:
reach_sample = reach_df.select("cpm").dropna().toPandas()

plt.figure(figsize=(10,5))
plt.hist(reach_sample["cpm"], bins=100)

plt.xlabel("CPM (Cost per 1000 Impressions)")
plt.ylabel("Number of Ads")
plt.title("Distribution of CPM Across Facebook Ads")

plt.show()